# Pseudo Label Generation — Midterm Retweet Graph
Generate per-user pseudo labels from hashtag usage, to use as a pretraining signal or downstream evaluation target.

In [ ]:
import glob
import pandas as pd
import numpy as np
import torch
from collections import defaultdict

CSV_PATTERN = "/project2/ll_774_951/midterm/*/*.csv"
SOURCE_GRAPH = "midterm/graph_co_retweet/graph_data.pt"
OUTPUT_PATH  = "midterm/graph_retweet/graph_data_pseudo.pt"

COLS = ['userid', 'hashtag', 'rt_hashtag', 'text', 'location', 'description']

## 1. Load CSVs

In [ ]:
files = sorted(glob.glob(CSV_PATTERN))
print(f"Found {len(files)} CSV files")

chunks = []
skipped = 0
for f in files:
    try:
        chunks.append(pd.read_csv(f, usecols=COLS, low_memory=False, on_bad_lines='skip'))
    except Exception as e:
        print(f"Skipping {f}: {e}")
        skipped += 1
df = pd.concat(chunks, ignore_index=True)
df['userid'] = pd.to_numeric(df['userid'], errors='coerce')
df = df.dropna(subset=['userid']).copy()
df['userid'] = df['userid'].astype(int)
print(f"Skipped files: {skipped}")
print(f"Total rows: {len(df):,}  |  Unique users: {df['userid'].nunique():,}")

## 2. Define label groups via hashtags
Each group is a label. A user is assigned a label if they (or their retweets) use any hashtag in that group.
Edit these lists to define your pseudo label taxonomy.

In [ ]:
# --- EDIT THIS ---
LABEL_GROUPS = {
    "pro_republican": [
        "maga", "trump2024", "letsgobrandon", "fjb", "redwave",
        "republicanparty", "gop", "trump", "saveamerica",
    ],
    "pro_democrat": [
        "voteblue", "democraticparty", "biden2024", "bluewave",
        "democrats", "votedem", "bidensamerica",
    ],
    "abortion_rights": [
        "prochoice", "mybodymychoice", "roevwade", "abortionrights",
        "bansoffdourbodies", "reproductiverights",
    ],
    "anti_abortion": [
        "prolife", "endabortion", "prolifegeneration", "marchforlife",
    ],
    # add more groups as needed...
}
# -----------------

label_names = list(LABEL_GROUPS.keys())
print("Labels:", label_names)

## 3. Score users per label group

In [ ]:
def parse_hashtags(series):
    """Extract lowercase hashtags from a column that may contain lists like ['tag1', 'tag2']."""
    out = []
    for val in series:
        if pd.isna(val):
            out.append(set())
            continue
        val = str(val).lower().replace("'", "").replace('"', '').strip("[]")
        out.append(set(t.strip() for t in val.split(",") if t.strip()))
    return out

df['_tags_own'] = parse_hashtags(df['hashtag'])
df['_tags_rt']  = parse_hashtags(df['rt_hashtag'])
df['_all_tags'] = df['_tags_own'] + df['_tags_rt']  # union per row

# Count hashtag hits per user per label
user_scores = defaultdict(lambda: defaultdict(int))
for _, row in df.iterrows():
    uid = row['userid']
    for label, tags in LABEL_GROUPS.items():
        hits = sum(1 for t in tags if t in row['_all_tags'])
        if hits:
            user_scores[uid][label] += hits

print(f"Users with at least one label hit: {len(user_scores):,}")

## 4. Assign labels (winner-takes-all, with min-score threshold)

In [ ]:
MIN_SCORE = 2  # user must have at least this many hashtag hits to get a label

label2idx = {name: i for i, name in enumerate(label_names)}
uid_to_label = {}

for uid, scores in user_scores.items():
    best_label = max(scores, key=scores.get)
    if scores[best_label] >= MIN_SCORE:
        uid_to_label[uid] = label2idx[best_label]

label_counts = pd.Series(uid_to_label.values()).value_counts().sort_index()
for idx, count in label_counts.items():
    print(f"  {label_names[idx]:30s}: {count:,} users")
print(f"\nTotal labeled: {len(uid_to_label):,}")

## 5. Map onto graph node indices and save

In [ ]:
raw = torch.load(SOURCE_GRAPH, map_location='cpu')
user_ids = raw['user_ids']
num_nodes = len(user_ids)
print(f"Graph nodes: {num_nodes:,}")

# Build node-level label tensor (-1 = unlabeled)
y_pseudo = torch.full((num_nodes,), -1, dtype=torch.long)
labeled = 0
for node_idx, uid in enumerate(user_ids):
    try:
        uid_int = int(uid)
    except (ValueError, TypeError):
        continue
    if uid_int in uid_to_label:
        y_pseudo[node_idx] = uid_to_label[uid_int]
        labeled += 1

print(f"Labeled nodes: {labeled:,} / {num_nodes:,} ({100*labeled/num_nodes:.1f}%)")

# Save as a new graph_data.pt with pseudo labels swapped in for y
out = dict(raw)
out['y'] = y_pseudo
out['label_names'] = label_names
torch.save(out, OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")